# Fold-safe severity evaluation

Use a stratified 42/18 split with seed 42. Fit TF-IDF only on the training fold and report class-specific results. The separate danger-term escalation is not included in these baseline classifier metrics.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "ResQ_Kerala_Complete_Project_Discussion.md").is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from backend.ai.severity_model import load_examples, new_pipeline, DISCLAIMER
rows = load_examples()
print(DISCLAIMER)


Synthetic training examples; uncalibrated demonstration score; admin verification required.


In [2]:
import csv
import json
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
train_rows, test_rows = train_test_split(rows, test_size=0.30, random_state=42,
                                       stratify=[row["severity"] for row in rows])
pipeline = new_pipeline().fit([row["message"] for row in train_rows], [row["severity"] for row in train_rows])
predicted = pipeline.predict([row["message"] for row in test_rows])
classes = ["LOW", "MODERATE", "HIGH"]
metrics = classification_report([row["severity"] for row in test_rows], predicted,
                                labels=classes, output_dict=True, zero_division=0)
print(json.dumps({"training_count": len(train_rows), "held_out_count": len(test_rows),
                  "accuracy": metrics["accuracy"], "per_class": {label: metrics[label] for label in classes},
                  "confusion_matrix": confusion_matrix([row["severity"] for row in test_rows], predicted, labels=classes).tolist()}, indent=2))
processed = PROJECT_ROOT / "data" / "processed"
processed.mkdir(exist_ok=True)
for name, records in (("train.csv", train_rows), ("test.csv", test_rows)):
    with (processed / name).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(records)
with (processed / "held_out_predictions.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["message", "expected", "predicted"])
    writer.writeheader()
    writer.writerows({"message": row["message"], "expected": row["severity"], "predicted": str(label)}
                    for row, label in zip(test_rows, predicted))
assert len(train_rows) == 42 and len(test_rows) == 18


{
  "training_count": 42,
  "held_out_count": 18,
  "accuracy": 0.6666666666666666,
  "per_class": {
    "LOW": {
      "precision": 0.6666666666666666,
      "recall": 0.6666666666666666,
      "f1-score": 0.6666666666666666,
      "support": 6.0
    },
    "MODERATE": {
      "precision": 0.6,
      "recall": 0.5,
      "f1-score": 0.5454545454545454,
      "support": 6.0
    },
    "HIGH": {
      "precision": 0.7142857142857143,
      "recall": 0.8333333333333334,
      "f1-score": 0.7692307692307693,
      "support": 6.0
    }
  },
  "confusion_matrix": [
    [
      4,
      2,
      0
    ],
    [
      1,
      3,
      2
    ],
    [
      1,
      0,
      5
    ]
  ]
}


Each class has only six held-out examples. Accuracy and probability scores are not operational risk validation. Inspect the errors, source representative reviewed messages, and recalibrate before using scores in operational decisions.